# Thai-Depression — FAST 5 Seeds × 20 Epochs (Frozen WangchanBERTa)

**แผนสำรองสำหรับ deadline**

- Official `train.json / valid.json / test.json`
- WangchanBERTa encoder **frozen**
- X-Transformers decoder + Attention Pooling + classifier
- ใช้ **fixed best parameters จาก Optuna เดิม**
- **ไม่รัน Optuna ใหม่**
- 5 independent runs: seeds `[13, 21, 42, 87, 123]`
- 20 epochs / run
- Mean / Median / Variance / Standard Deviation
- Precision / Recall / F1 / Support แยก `no_depression` และ `depression`
- ถ้า Kaggle มี **T4 ×2** จะใช้ `torch.nn.DataParallel` อัตโนมัติ

> หมายเหตุเชิงวิธีวิจัย: best params ชุดนี้ได้จาก optimized notebook เดิมก่อนเปลี่ยน encoder เป็น frozen
> จึงเป็น **fast fallback protocol** ไม่ใช่การ re-optimize hyperparameters ภายใต้ frozen setup
> แต่ทั้ง 5 repeated runs ใช้ hyperparameters ชุดเดียวกันทั้งหมดเพื่อวัด seed variance.

In [1]:
!pip install -q "transformers>=4.44,<5" sentencepiece x-transformers scikit-learn pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.1/143.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.8 MB/s eta 0:00:00


In [2]:
import os, glob, gc, json, random, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from x_transformers import Decoder

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

PRIMARY_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
GPU_COUNT = torch.cuda.device_count()

print("DEVICE:", PRIMARY_DEVICE)
print("GPU COUNT:", GPU_COUNT)
for i in range(GPU_COUNT):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

USE_DATA_PARALLEL = GPU_COUNT >= 2
print("USE_DATA_PARALLEL:", USE_DATA_PARALLEL)

DEVICE: cuda:0
GPU COUNT: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
USE_DATA_PARALLEL: True


In [3]:
# =========================================================
# CONFIG
# =========================================================

MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

MAX_LEN = 128
BATCH_SIZE = 64
CACHE_BATCH_SIZE = 32

EPOCHS = 20
TRAIN_SEEDS = [13, 21, 42, 87, 123]

LABEL_NAMES = ["no_depression", "depression"]
NUM_LABELS = 2

# Best params จาก notebook:
# wanchanberta-x-transformers-thai-depression-optim(6).ipynb
FIXED_BEST_PARAMS = {
    "depth": 2,
    "heads": 4,
    "attn_dropout": 0.2240577833126539,
    "ff_dropout": 0.4922163075549199,
    "ff_mult": 4,
    "lr": 3.0331847361552553e-05,
    "weight_decay": 0.0002821544949248052,
}

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("EPOCHS:", EPOCHS)
print("SEEDS:", TRAIN_SEEDS)
print("BEST PARAMS:", FIXED_BEST_PARAMS)

EPOCHS: 20
SEEDS: [13, 21, 42, 87, 123]
BEST PARAMS: {'depth': 2, 'heads': 4, 'attn_dropout': 0.2240577833126539, 'ff_dropout': 0.4922163075549199, 'ff_mult': 4, 'lr': 3.0331847361552553e-05, 'weight_decay': 0.0002821544949248052}


In [4]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [5]:
# =========================================================
# FIND OFFICIAL DATASET FILES
# =========================================================

def find_json(filename):
    candidates = [
        f"/content/{filename}",
        f"/mnt/data/{filename}",
    ]
    for p in candidates:
        if os.path.exists(p):
            return p

    hits = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    dep_hits = [p for p in hits if "depression" in p.lower()]
    if dep_hits:
        return dep_hits[0]
    if hits:
        return hits[0]
    raise FileNotFoundError(f"ไม่พบ {filename}")

TRAIN_PATH = find_json("train.json")
VALID_PATH = find_json("valid.json")
TEST_PATH  = find_json("test.json")

print(TRAIN_PATH)
print(VALID_PATH)
print(TEST_PATH)

/kaggle/input/datasets/abdulhakimsamae/thai-depression/Thai Depression/train.json
/kaggle/input/datasets/abdulhakimsamae/thai-depression/Thai Depression/valid.json
/kaggle/input/datasets/abdulhakimsamae/thai-depression/Thai Depression/test.json


In [6]:
# =========================================================
# LOAD DATA — preprocessing ตาม notebook เดิม
# =========================================================

def load_json_dataset(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    texts, labels = [], []

    for item in raw:
        if not isinstance(item, (list, tuple)) or len(item) != 2:
            continue

        text, label = item
        if not isinstance(text, str):
            continue

        text = text.replace("---- “ ----", " ").replace("\n", " ")
        text = " ".join(text.split()).strip()

        if len(text) < 2:
            continue

        if label == "no_depression":
            y = 0
        elif label == "depression":
            y = 1
        else:
            continue

        texts.append(text)
        labels.append(y)

    return pd.DataFrame({"text": texts, "label": labels})

train_df = load_json_dataset(TRAIN_PATH)
val_df   = load_json_dataset(VALID_PATH)
test_df  = load_json_dataset(TEST_PATH)

print("TRAIN:", len(train_df))
print("VALID:", len(val_df))
print("TEST :", len(test_df))

print("\nTRAIN LABELS")
print(train_df["label"].value_counts().sort_index())

TRAIN: 25004
VALID: 3335
TEST : 5004

TRAIN LABELS
label
0    12239
1    12765
Name: count, dtype: int64


In [7]:
# =========================================================
# CLASS WEIGHTS — สูตรเดียวกับ notebook เดิม
# =========================================================

class_counts = train_df["label"].value_counts().sort_index().values.astype(np.float32)
class_weights_np = 1.0 / class_counts
class_weights_np = class_weights_np / class_weights_np.sum()

print("CLASS COUNTS:", class_counts)
print("CLASS WEIGHTS:", class_weights_np)

CLASS COUNTS: [12239. 12765.]
CLASS WEIGHTS: [0.5105183  0.48948172]


In [8]:
# =========================================================
# FROZEN ENCODER + RAM CACHE
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

encoder = AutoModel.from_pretrained(MODEL_NAME).to(PRIMARY_DEVICE)
for p in encoder.parameters():
    p.requires_grad = False
encoder.eval()

HIDDEN_SIZE = encoder.config.hidden_size

class RawDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame["text"].tolist()
        self.labels = frame["label"].values.astype(np.int64)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


@torch.no_grad()
def precompute_to_ram(frame, name):
    ds = RawDataset(frame)
    loader = DataLoader(
        ds,
        batch_size=CACHE_BATCH_SIZE,
        shuffle=False,
        pin_memory=torch.cuda.is_available()
    )

    n = len(ds)

    # preallocate ป้องกัน peak RAM จาก list + torch.cat
    hidden_store = torch.empty(
        (n, MAX_LEN, HIDDEN_SIZE),
        dtype=torch.float16
    )
    mask_store = torch.empty(
        (n, MAX_LEN),
        dtype=torch.bool
    )
    label_store = torch.empty(
        (n,),
        dtype=torch.long
    )

    cursor = 0
    use_amp = PRIMARY_DEVICE.type == "cuda"

    for batch in tqdm(loader, desc=f"Encoding {name}"):
        ids = batch["input_ids"].to(PRIMARY_DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(PRIMARY_DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            h = encoder(
                input_ids=ids,
                attention_mask=mask
            ).last_hidden_state

        b = h.size(0)
        hidden_store[cursor:cursor+b].copy_(h.half().cpu())
        mask_store[cursor:cursor+b].copy_(mask.bool().cpu())
        label_store[cursor:cursor+b].copy_(batch["labels"])
        cursor += b

    return hidden_store, mask_store, label_store


train_hidden, train_masks, train_labels = precompute_to_ram(train_df, "train")
val_hidden,   val_masks,   val_labels   = precompute_to_ram(val_df, "validation")
test_hidden,  test_masks,  test_labels  = precompute_to_ram(test_df, "test")

del encoder
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("TRAIN CACHE:", train_hidden.shape, train_hidden.dtype)

tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/423M [00:00<?, ?B/s]

Encoding train:   0%|          | 0/782 [00:00<?, ?it/s]

Encoding validation:   0%|          | 0/105 [00:00<?, ?it/s]

Encoding test:   0%|          | 0/157 [00:00<?, ?it/s]

TRAIN CACHE: torch.Size([25004, 128, 768]) torch.float16


In [9]:
# =========================================================
# CACHED DATASET
# =========================================================

class CachedDataset(Dataset):
    def __init__(self, hidden, masks, labels):
        self.hidden = hidden
        self.masks = masks
        self.labels = labels

    def __len__(self):
        return len(self.hidden)

    def __getitem__(self, idx):
        return {
            "hidden": self.hidden[idx],
            "attention_mask": self.masks[idx],
            "labels": self.labels[idx]
        }

train_dataset = CachedDataset(train_hidden, train_masks, train_labels)
val_dataset   = CachedDataset(val_hidden, val_masks, val_labels)
test_dataset  = CachedDataset(test_hidden, test_masks, test_labels)

def make_loaders(seed):
    g = torch.Generator()
    g.manual_seed(seed)

    common = dict(
        batch_size=BATCH_SIZE,
        pin_memory=torch.cuda.is_available(),
        num_workers=0
    )

    return (
        DataLoader(train_dataset, shuffle=True, generator=g, **common),
        DataLoader(val_dataset, shuffle=False, **common),
        DataLoader(test_dataset, shuffle=False, **common),
    )

In [10]:
# =========================================================
# MODEL
# =========================================================

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x, mask):
        scores = self.attention(x).squeeze(-1)
        scores = scores.masked_fill(
            ~mask.bool(),
            torch.finfo(scores.dtype).min
        )
        weights = torch.softmax(scores, dim=1)
        return torch.sum(x * weights.unsqueeze(-1), dim=1)


class EmotionModel(nn.Module):
    def __init__(self, depth, heads, attn_dropout, ff_dropout, ff_mult):
        super().__init__()

        self.decoder = Decoder(
            dim=HIDDEN_SIZE,
            depth=depth,
            heads=heads,
            attn_dropout=attn_dropout,
            ff_dropout=ff_dropout,
            ff_mult=ff_mult
        )

        self.pooling = AttentionPooling(HIDDEN_SIZE)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(HIDDEN_SIZE, NUM_LABELS)

    def forward(self, hidden, attention_mask):
        # ห้าม .to(cuda:0) ภายใน forward เพื่อให้ DataParallel scatter ไป GPU1 ได้
        x = self.decoder(hidden.float())
        x = self.pooling(x, attention_mask)
        x = self.dropout(x)
        return self.fc(x)


def build_model():
    p = FIXED_BEST_PARAMS

    model = EmotionModel(
        depth=p["depth"],
        heads=p["heads"],
        attn_dropout=p["attn_dropout"],
        ff_dropout=p["ff_dropout"],
        ff_mult=p["ff_mult"]
    ).to(PRIMARY_DEVICE)

    if USE_DATA_PARALLEL:
        model = nn.DataParallel(
            model,
            device_ids=list(range(GPU_COUNT))
        )
        print("DataParallel GPUs:", list(range(GPU_COUNT)))

    return model

In [11]:
# =========================================================
# EVALUATION
# =========================================================

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []

    for batch in loader:
        hidden = batch["hidden"].to(PRIMARY_DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(PRIMARY_DEVICE, non_blocking=True)
        labels = batch["labels"].to(PRIMARY_DEVICE, non_blocking=True)

        logits = model(hidden, mask)
        pred = torch.argmax(logits, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(pred.cpu().numpy())

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

    result["per_precision"] = precision_score(
        y_true, y_pred, labels=[0,1], average=None, zero_division=0
    )
    result["per_recall"] = recall_score(
        y_true, y_pred, labels=[0,1], average=None, zero_division=0
    )
    result["per_f1"] = f1_score(
        y_true, y_pred, labels=[0,1], average=None, zero_division=0
    )
    result["support"] = np.array([(y_true==0).sum(), (y_true==1).sum()])

    return result

In [12]:
# =========================================================
# TRAIN ONE SEED — 20 epochs
# ใช้ best validation Macro-F1 ภายใน 20 epochs ตาม model-selection เดิม
# =========================================================

def train_one_seed(seed):
    seed_everything(seed)

    train_loader, val_loader, test_loader = make_loaders(seed)
    model = build_model()

    class_weights = torch.tensor(
        class_weights_np,
        dtype=torch.float32,
        device=PRIMARY_DEVICE
    )

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=FIXED_BEST_PARAMS["lr"],
        weight_decay=FIXED_BEST_PARAMS["weight_decay"]
    )

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    use_amp = PRIMARY_DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_macro = -1.0
    best_state = None

    start = time.perf_counter()

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            hidden = batch["hidden"].to(PRIMARY_DEVICE, non_blocking=True)
            mask = batch["attention_mask"].to(PRIMARY_DEVICE, non_blocking=True)
            labels = batch["labels"].to(PRIMARY_DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(hidden, mask)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()

        val_result = evaluate(model, val_loader)

        print(
            f"seed={seed} | epoch {epoch+1:02d}/{EPOCHS} | "
            f"loss={total_loss/len(train_loader):.4f} | "
            f"val_acc={val_result['accuracy']:.4f} | "
            f"val_macro={val_result['macro_f1']:.4f}"
        )

        if val_result["macro_f1"] > best_macro:
            best_macro = val_result["macro_f1"]

            # save decoder state in RAM only
            state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            best_state = {k: v.detach().cpu().clone() for k, v in state.items()}

    target = model.module if isinstance(model, nn.DataParallel) else model
    target.load_state_dict(best_state)

    test_result = evaluate(model, test_loader)
    elapsed = time.perf_counter() - start

    del model, optimizer, scheduler, best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return test_result, best_macro, elapsed

In [13]:
# =========================================================
# RUN 5 SEEDS
# =========================================================

overall_rows = []
per_label_rows = []

for run_idx, seed in enumerate(TRAIN_SEEDS, start=1):
    print("\n" + "="*90)
    print(f"RUN {run_idx}/5 | seed={seed}")
    print("="*90)

    result, best_val_macro, elapsed = train_one_seed(seed)

    overall_rows.append({
        "run": run_idx,
        "seed": seed,
        "accuracy": result["accuracy"],
        "micro_precision": result["micro_precision"],
        "micro_recall": result["micro_recall"],
        "micro_f1": result["micro_f1"],
        "macro_precision": result["macro_precision"],
        "macro_recall": result["macro_recall"],
        "macro_f1": result["macro_f1"],
        "weighted_f1": result["weighted_f1"],
        "best_val_macro_f1": best_val_macro,
        "minutes": elapsed / 60.0,
    })

    for i, label in enumerate(LABEL_NAMES):
        per_label_rows.append({
            "run": run_idx,
            "seed": seed,
            "label": label,
            "precision": result["per_precision"][i],
            "recall": result["per_recall"][i],
            "f1": result["per_f1"][i],
            "support": int(result["support"][i]),
        })

results_df = pd.DataFrame(overall_rows)
per_label_df = pd.DataFrame(per_label_rows)

display(results_df)
display(per_label_df)


RUN 1/5 | seed=13
DataParallel GPUs: [0, 1]
seed=13 | epoch 01/20 | loss=0.6434 | val_acc=0.7511 | val_macro=0.7502
seed=13 | epoch 02/20 | loss=0.5217 | val_acc=0.7547 | val_macro=0.7530
seed=13 | epoch 03/20 | loss=0.4926 | val_acc=0.7778 | val_macro=0.7770
seed=13 | epoch 04/20 | loss=0.4709 | val_acc=0.7778 | val_macro=0.7778
seed=13 | epoch 05/20 | loss=0.4469 | val_acc=0.7823 | val_macro=0.7821
seed=13 | epoch 06/20 | loss=0.4311 | val_acc=0.7832 | val_macro=0.7830
seed=13 | epoch 07/20 | loss=0.4110 | val_acc=0.7808 | val_macro=0.7794
seed=13 | epoch 08/20 | loss=0.3955 | val_acc=0.7811 | val_macro=0.7808
seed=13 | epoch 09/20 | loss=0.3752 | val_acc=0.7865 | val_macro=0.7860
seed=13 | epoch 10/20 | loss=0.3577 | val_acc=0.7781 | val_macro=0.7750
seed=13 | epoch 11/20 | loss=0.3440 | val_acc=0.7907 | val_macro=0.7898
seed=13 | epoch 12/20 | loss=0.3252 | val_acc=0.7838 | val_macro=0.7838
seed=13 | epoch 13/20 | loss=0.3064 | val_acc=0.7844 | val_macro=0.7841
seed=13 | epoch 14/

,run,seed,accuracy,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,weighted_f1,best_val_macro_f1,minutes
0,1,13,0.776379,0.776379,0.776379,0.776379,0.781569,0.774820,0.774596,0.775029,0.789765,6.363239
1,2,21,0.776779,0.776779,0.776779,0.776779,0.781728,0.775255,0.775063,0.775487,0.783670,6.299166
2,3,42,0.772582,0.772582,0.772582,0.772582,0.774206,0.771639,0.771751,0.772048,0.786451,6.300939
3,4,87,0.776978,0.776978,0.776978,0.776978,0.784850,0.775097,0.774544,0.775049,0.779880,6.282864
4,5,123,0.774380,0.774380,0.774380,0.774380,0.776658,0.773296,0.773365,0.773692,0.788656,6.344841


,run,seed,label,precision,recall,f1,support
0,1,13,no_depression,0.814780,0.702614,0.754551,2448
1,1,13,depression,0.748358,0.847027,0.794641,2556
2,2,21,no_depression,0.814063,0.704657,0.755419,2448
3,2,21,depression,0.749393,0.845853,0.794707,2556
4,3,42,no_depression,0.790594,0.727941,0.757975,2448
5,3,42,depression,0.757818,0.815336,0.785526,2556
6,4,87,no_depression,0.827112,0.687908,0.751115,2448
7,4,87,depression,0.742588,0.862285,0.797972,2556
8,5,123,no_depression,0.796938,0.723039,0.758192,2448
9,5,123,depression,0.756378,0.823552,0.788537,2556


In [14]:
# =========================================================
# STATISTICAL SUMMARY
# =========================================================

metric_cols = [
    "accuracy",
    "micro_precision",
    "micro_recall",
    "micro_f1",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
]

summary_df = pd.DataFrame([
    {
        "metric": m,
        "mean": results_df[m].mean(),
        "median": results_df[m].median(),
        "variance": results_df[m].var(ddof=1),
        "std": results_df[m].std(ddof=1),
        "min": results_df[m].min(),
        "max": results_df[m].max(),
    }
    for m in metric_cols
])

per_label_summary = (
    per_label_df.groupby("label", sort=False)
    .agg(
        precision_mean=("precision","mean"),
        precision_variance=("precision", lambda x: x.var(ddof=1)),
        precision_std=("precision", lambda x: x.std(ddof=1)),
        recall_mean=("recall","mean"),
        recall_variance=("recall", lambda x: x.var(ddof=1)),
        recall_std=("recall", lambda x: x.std(ddof=1)),
        f1_mean=("f1","mean"),
        f1_median=("f1","median"),
        f1_variance=("f1", lambda x: x.var(ddof=1)),
        f1_std=("f1", lambda x: x.std(ddof=1)),
    )
    .reset_index()
)

print("5-RUN SUMMARY")
display(summary_df)

print("PER-LABEL SUMMARY")
display(per_label_summary)

5-RUN SUMMARY


,metric,mean,median,variance,std,min,max
0,accuracy,0.775420,0.776379,0.000004,0.001893,0.772582,0.776978
1,micro_precision,0.775420,0.776379,0.000004,0.001893,0.772582,0.776978
2,micro_recall,0.775420,0.776379,0.000004,0.001893,0.772582,0.776978
3,micro_f1,0.775420,0.776379,0.000004,0.001893,0.772582,0.776978
4,macro_precision,0.779802,0.781569,0.000018,0.004287,0.774206,0.784850
5,macro_recall,0.774021,0.774820,0.000002,0.001543,0.771639,0.775255
6,macro_f1,0.773864,0.774544,0.000002,0.001337,0.771751,0.775063
7,weighted_f1,0.774261,0.775029,0.000002,0.001409,0.772048,0.775487


PER-LABEL SUMMARY


,label,precision_mean,precision_variance,precision_std,recall_mean,recall_variance,recall_std,f1_mean,f1_median,f1_variance,f1_std
0,no_depression,0.808698,0.000218,0.014755,0.709232,0.000265,0.016280,0.755451,0.755419,0.000008,0.002894
1,depression,0.750907,0.000039,0.006239,0.838811,0.000363,0.019052,0.792277,0.794641,0.000026,0.005087


In [15]:
# =========================================================
# EXPORT
# =========================================================

files = {
    "runs": os.path.join(OUTPUT_DIR, "thai_depression_FAST_5seeds_20epochs_runs.csv"),
    "summary": os.path.join(OUTPUT_DIR, "thai_depression_FAST_5seeds_20epochs_summary.csv"),
    "per_label": os.path.join(OUTPUT_DIR, "thai_depression_FAST_5seeds_20epochs_per_label.csv"),
    "per_label_summary": os.path.join(OUTPUT_DIR, "thai_depression_FAST_5seeds_20epochs_per_label_summary.csv"),
}

results_df.to_csv(files["runs"], index=False, encoding="utf-8-sig")
summary_df.to_csv(files["summary"], index=False, encoding="utf-8-sig")
per_label_df.to_csv(files["per_label"], index=False, encoding="utf-8-sig")
per_label_summary.to_csv(files["per_label_summary"], index=False, encoding="utf-8-sig")

for k,v in files.items():
    print(k, "->", v)

runs -> /kaggle/working/thai_depression_FAST_5seeds_20epochs_runs.csv
summary -> /kaggle/working/thai_depression_FAST_5seeds_20epochs_summary.csv
per_label -> /kaggle/working/thai_depression_FAST_5seeds_20epochs_per_label.csv
per_label_summary -> /kaggle/working/thai_depression_FAST_5seeds_20epochs_per_label_summary.csv
